## 1. Mount Google Drive

In [1]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

# -------------------------
# Persistent files on Drive
# -------------------------
DRIVE_ZIP = Path("/content/drive/MyDrive/T2ICount-assets.zip")

# Keep training checkpoints persistent on Drive
DRIVE_ASSET_ROOT = "/content/drive/MyDrive/T2ICount-assets"
DRIVE_CHECKPOINT_ROOT = (
    Path(DRIVE_ASSET_ROOT) / "checkpoints/baseline_retrain"
)
DRIVE_CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)

# -------------------------
# Fast local Colab storage
# -------------------------
LOCAL_ZIP = Path("/content/T2ICount-assets.zip")

# IMPORTANT:
# ASSET_ROOT now points to Colab local SSD, NOT Google Drive.
ASSET_ROOT = "/content/T2ICount-assets"
ASSET_ROOT_PATH = Path(ASSET_ROOT)

REPO_DIR = "/content/T2ICount-Implementation"

# Infrastructure smoke tests are already validated. Keep this False for
# normal experiments; enable only after infrastructure changes.
RUN_INFRA_SMOKE_TESTS = False

print("Drive ZIP :", DRIVE_ZIP)
print("Local assets:", ASSET_ROOT)
print("Drive saves :", DRIVE_ASSET_ROOT)

Mounted at /content/drive
Drive ZIP : /content/drive/MyDrive/T2ICount-assets.zip
Local assets: /content/T2ICount-assets
Drive saves : /content/drive/MyDrive/T2ICount-assets


In [2]:
import shutil
import subprocess
import time
import zipfile
from pathlib import Path

from tqdm.auto import tqdm


# ============================================================
# Required assets
# ============================================================

REQUIRED_ASSETS = {
    "CLIP": (
        ASSET_ROOT_PATH
        / "pretrained/clip-vit-large-patch14",
        "directory",
    ),

    "Stable Diffusion": (
        ASSET_ROOT_PATH
        / "pretrained/sd-v1-5/v1-5-pruned-emaonly.ckpt",
        "file",
    ),

    "FSC147": (
        ASSET_ROOT_PATH
        / "datasets/FSC147",
        "directory",
    ),

    "T2ICount checkpoint": (
        ASSET_ROOT_PATH
        / "checkpoints/official/best_model_paper.pth",
        "file",
    ),
}


def check_assets(verbose=True):
    status = {}
    for name, (path, kind) in REQUIRED_ASSETS.items():
        status[name] = path.is_dir() if kind == "directory" else path.is_file()

    if verbose:
        print("\nAsset status:")

        for name, ok in status.items():
            tag = "OK" if ok else "MISSING"

            print(f"  [{tag}] {name}")

            if not ok:
                print(f"       {REQUIRED_ASSETS[name][0]}")

    return all(status.values())


def human_size(num_bytes):
    units = ["B", "KB", "MB", "GB", "TB"]

    size = float(num_bytes)

    for unit in units:
        if size < 1024:
            return f"{size:.2f} {unit}"

        size /= 1024

    return f"{size:.2f} PB"


def copy_with_progress(src, dst, chunk_size=64 * 1024 * 1024):
    """
    Copy one large file from Drive -> local SSD with progress.
    """

    total = src.stat().st_size

    dst.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    with src.open("rb") as fsrc, dst.open("wb") as fdst:
        with tqdm(
            total=total,
            unit="B",
            unit_scale=True,
            unit_divisor=1024,
            desc="Copy ZIP Drive -> local",
        ) as progress:

            while True:
                chunk = fsrc.read(chunk_size)

                if not chunk:
                    break

                fdst.write(chunk)
                progress.update(len(chunk))


# ============================================================
# 1. Skip everything if local assets are already complete
# ============================================================

if check_assets():
    print(
        "\nLocal assets already complete:",
        ASSET_ROOT_PATH,
    )

else:

    # ========================================================
    # 2. ZIP must exist on Drive
    # ========================================================

    if not DRIVE_ZIP.is_file():
        raise FileNotFoundError(
            "\nCannot find asset ZIP on Google Drive:\n"
            f"{DRIVE_ZIP}\n\n"
            "Upload T2ICount-assets.zip to My Drive first."
        )

    drive_zip_size = DRIVE_ZIP.stat().st_size

    print("\nDrive ZIP found.")
    print("ZIP size:", human_size(drive_zip_size))


    # ========================================================
    # 3. Reuse local ZIP if it was already copied successfully
    # ========================================================

    local_zip_valid = (
        LOCAL_ZIP.is_file()
        and LOCAL_ZIP.stat().st_size == drive_zip_size
    )

    if local_zip_valid:
        print(
            "\nLocal ZIP already exists with matching size."
        )
        print("Skipping Drive copy:", LOCAL_ZIP)

    else:

        # Remove ONLY an incomplete LOCAL ZIP.
        # Never deletes anything from Google Drive.
        if LOCAL_ZIP.exists():
            print(
                "\nRemoving incomplete local ZIP:",
                LOCAL_ZIP,
            )
            LOCAL_ZIP.unlink()

        free_space = shutil.disk_usage("/content").free

        print(
            "Free local disk:",
            human_size(free_space),
        )

        if free_space < drive_zip_size + 5 * 1024**3:
            raise RuntimeError(
                "Not enough local disk space to safely copy "
                "the asset archive."
            )

        print(
            "\nCopying ZIP from Google Drive to Colab SSD..."
        )

        start = time.perf_counter()

        copy_with_progress(
            DRIVE_ZIP,
            LOCAL_ZIP,
        )

        elapsed = time.perf_counter() - start

        print(
            f"\nZIP copied in {elapsed / 60:.1f} minutes."
        )


    # ========================================================
    # 4. Validate ZIP
    # ========================================================

    print("\nChecking ZIP integrity...")

    if not zipfile.is_zipfile(LOCAL_ZIP):
        raise RuntimeError(
            f"Local archive is not a valid ZIP:\n{LOCAL_ZIP}"
        )


    # ========================================================
    # 5. Determine archive layout
    # ========================================================

    with zipfile.ZipFile(LOCAL_ZIP, "r") as zf:

        members = zf.infolist()

        names = [
            info.filename.replace("\\", "/")
            for info in members
        ]

        has_root_folder = any(
            name.startswith("T2ICount-assets/")
            for name in names
        )

        total_uncompressed = sum(
            info.file_size
            for info in members
        )

    print(
        "Uncompressed size:",
        human_size(total_uncompressed),
    )

    print(
        "Files in archive:",
        len(members),
    )


    if has_root_folder:

        # ZIP structure:
        #
        # T2ICount-assets/
        # ├── pretrained/
        # ├── datasets/
        # └── checkpoints/
        #
        # Extract to /content

        extract_to = Path("/content")

    else:

        # ZIP structure:
        #
        # pretrained/
        # datasets/
        # checkpoints/
        #
        # Extract directly to asset root

        ASSET_ROOT_PATH.mkdir(
            parents=True,
            exist_ok=True,
        )

        extract_to = ASSET_ROOT_PATH


    print(
        "\nArchive layout:",
        (
            "contains T2ICount-assets/"
            if has_root_folder
            else "contents start inside asset root"
        ),
    )

    print("Extract target:", extract_to)


    # ========================================================
    # 6. Extract on LOCAL SSD
    # ========================================================

    print(
        "\nExtracting on Colab local SSD..."
    )
    print(
        "This should be much faster than extracting "
        "directly into Google Drive."
    )

    start = time.perf_counter()

    # Use system unzip: significantly better suited for
    # extracting many files locally than doing Drive FUSE I/O.
    result = subprocess.run(
        [
            "unzip",
            "-q",
            "-o",
            str(LOCAL_ZIP),
            "-d",
            str(extract_to),
        ],
        text=True,
    )

    if result.returncode != 0:
        raise RuntimeError(
            f"unzip failed with return code "
            f"{result.returncode}"
        )

    elapsed = time.perf_counter() - start

    print(
        f"Extraction finished in "
        f"{elapsed / 60:.1f} minutes."
    )


    # ========================================================
    # 7. Validate extracted assets
    # ========================================================

    print("\nValidating extracted assets...")

    if not check_assets():
        raise RuntimeError(
            "\nExtraction finished but required assets "
            "are still missing.\n"
            "Do NOT continue to model validation/training."
        )


    # ========================================================
    # 8. Free local space
    # ========================================================

    # ZIP is no longer needed after successful extraction.
    if LOCAL_ZIP.exists():
        print(
            "\nRemoving local ZIP to recover disk space..."
        )
        LOCAL_ZIP.unlink()


    print("\n========================================")
    print("LOCAL ASSET BOOTSTRAP PASSED")
    print("========================================")
    print("Runtime assets :", ASSET_ROOT_PATH)
    print("Checkpoints    :", DRIVE_CHECKPOINT_ROOT)


Asset status:
  [MISSING] CLIP
       /content/T2ICount-assets/pretrained/clip-vit-large-patch14
  [MISSING] Stable Diffusion
       /content/T2ICount-assets/pretrained/sd-v1-5/v1-5-pruned-emaonly.ckpt
  [MISSING] FSC147
       /content/T2ICount-assets/datasets/FSC147
  [MISSING] T2ICount checkpoint
       /content/T2ICount-assets/checkpoints/official/best_model_paper.pth

Drive ZIP found.
ZIP size: 12.96 GB
Free local disk: 65.43 GB

Copying ZIP from Google Drive to Colab SSD...


Copy ZIP Drive -> local:   0%|          | 0.00/13.0G [00:00<?, ?B/s]


ZIP copied in 7.6 minutes.

Checking ZIP integrity...
Uncompressed size: 17.86 GB
Files in archive: 15781

Archive layout: contains T2ICount-assets/
Extract target: /content

Extracting on Colab local SSD...
This should be much faster than extracting directly into Google Drive.
Extraction finished in 3.0 minutes.

Validating extracted assets...

Asset status:
  [OK] CLIP
  [OK] Stable Diffusion
  [OK] FSC147
  [OK] T2ICount checkpoint

Removing local ZIP to recover disk space...

LOCAL ASSET BOOTSTRAP PASSED
Runtime assets : /content/T2ICount-assets
Checkpoints    : /content/drive/MyDrive/T2ICount-assets/checkpoints/baseline_retrain


## 2. Clone or update the repository

In [3]:
import os
import pathlib
import subprocess

REPO_URL = "https://github.com/bqa100507-spec/T2ICount-Implementation.git"
repo_path = pathlib.Path(REPO_DIR)
if not repo_path.exists():
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
elif not (repo_path / ".git").is_dir():
    raise RuntimeError(f"{REPO_DIR} exists but is not a Git repository.")
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
os.chdir(REPO_DIR)
print("Repository:", pathlib.Path.cwd())

Repository: /content/T2ICount-Implementation


In [4]:
%cd /content/T2ICount-Implementation

!git fetch origin
!git checkout DUMLO
!git pull --ff-only origin DUMLO
!git status --short --branch

/content
Branch 'DUMLO' set up to track remote branch 'DUMLO' from 'origin'.
Switched to a new branch 'DUMLO'
From https://github.com/bqa100507-spec/T2ICount-Implementation
 * branch            DUMLO      -> FETCH_HEAD
Already up to date.
## DUMLO...origin/DUMLO


## 3. Install Colab dependencies
If installation replaces important runtime packages such as PyTorch, Transformers, or Lightning, restart the runtime/kernel before continuing when required. Do not continue validation with a partially reloaded environment.

In [5]:
%pip install -r requirements-colab.txt

Obtaining taming-transformers from git+https://github.com/CompVis/taming-transformers.git@3ba01b241669f5ade541ce990f7650a3b8f65318#egg=taming-transformers (from -r requirements-colab.txt (line 14))
  Cloning https://github.com/CompVis/taming-transformers.git (to revision 3ba01b241669f5ade541ce990f7650a3b8f65318) to ./src/taming-transformers
  Running command git clone --filter=blob:none --quiet https://github.com/CompVis/taming-transformers.git /content/T2ICount-Implementation/src/taming-transformers
  Running command git rev-parse -q --verify 'sha^3ba01b241669f5ade541ce990f7650a3b8f65318'
  Running command git fetch -q https://github.com/CompVis/taming-transformers.git 3ba01b241669f5ade541ce990f7650a3b8f65318
  Resolved https://github.com/CompVis/taming-transformers.git to commit 3ba01b241669f5ade541ce990f7650a3b8f65318
  Preparing metadata (setup.py) ... done
Obtaining clip from git+https://github.com/openai/CLIP.git@d05afc436d78f1c48dc0dbf8e5980a9d471f35f6#egg=clip (from -r requirem

## 4. Configure the notebook Python environment

In [6]:
import os

os.environ["T2ICOUNT_ASSET_ROOT"] = ASSET_ROOT
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
print("T2ICOUNT_ASSET_ROOT=", os.environ["T2ICOUNT_ASSET_ROOT"] )

T2ICOUNT_ASSET_ROOT= /content/T2ICount-assets


## 5. Preflight + path

In [7]:
from pathlib import Path
import subprocess
import sys
import time

REPO_ROOT = Path("/content/T2ICount-Implementation")
ASSET_ROOT = "/content/T2ICount-assets"
DRIVE_ASSET_ROOT = "/content/drive/MyDrive/T2ICount-assets"

# Original DUMLO 1000x10, best observed checkpoint at epoch 7
DUMLO_E7 = Path(
    DRIVE_ASSET_ROOT
) / (
    "checkpoints/dumlo_retrain/pilot_1000x10/"
    "dumlo_1000x10/best_model_7.pth"
)

OUT_DIR = Path(
    DRIVE_ASSET_ROOT
) / "outputs/dumlo_gradient_diagnostics"

OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Repo:", REPO_ROOT)
print("Checkpoint:", DUMLO_E7)
print("Checkpoint exists:", DUMLO_E7.exists())
print("Output:", OUT_DIR)

assert REPO_ROOT.exists(), REPO_ROOT
assert DUMLO_E7.exists(), DUMLO_E7

# Record exact code state used for the diagnostic.
subprocess.run(
    ["git", "status", "--short", "--branch"],
    cwd=REPO_ROOT,
    check=True,
)

subprocess.run(
    ["git", "rev-parse", "HEAD"],
    cwd=REPO_ROOT,
    check=True,
)

Repo: /content/T2ICount-Implementation
Checkpoint: /content/drive/MyDrive/T2ICount-assets/checkpoints/dumlo_retrain/pilot_1000x10/dumlo_1000x10/best_model_7.pth
Checkpoint exists: True
Output: /content/drive/MyDrive/T2ICount-assets/outputs/dumlo_gradient_diagnostics


CompletedProcess(args=['git', 'rev-parse', 'HEAD'], returncode=0)

## 6. CLI check

In [8]:
cmd = [
    sys.executable,
    "tools/diagnose_dumlo_gradients.py",
    "--help",
]

result = subprocess.run(
    cmd,
    cwd=REPO_ROOT,
    text=True,
)

print("RETURN CODE:", result.returncode)
assert result.returncode == 0

RETURN CODE: 0


In [9]:
from pathlib import Path

# ============================================================
# Checkpoints
# ============================================================

CKPT_DUMLO = (
    Path(DRIVE_ASSET_ROOT)
    / "checkpoints/dumlo_retrain/pilot_1000x10"
    / "dumlo_1000x10"
    / "best_model_7.pth"
)

CKPT_COUNT_LOW = (
    Path(DRIVE_ASSET_ROOT)
    / "checkpoints/dumlo_retrain/lambda_count_01_1000x10"
    / "dumlo_lc01_1000x10"
    / "best_model_7.pth"
)

CKPT_OT_LOW = (
    Path(DRIVE_ASSET_ROOT)
    / "checkpoints/dumlo_retrain/lambda_ot_001_1000x10"
    / "dumlo_lot001_1000x10"
    / "best_model_7.pth"
)

CKPT_BASELINE = (
    Path(DRIVE_ASSET_ROOT)
    / "checkpoints/baseline_retrain/pilot_1000x10"
    / "baseline_1000x10"
    / "best_model_7.pth"
)

checkpoints = {
    "DUMLO original": CKPT_DUMLO,
    "Count-low": CKPT_COUNT_LOW,
    "OT-low": CKPT_OT_LOW,
    "Baseline": CKPT_BASELINE,
}

for name, path in checkpoints.items():
    print(f"{name:16s}: {path}")
    print("  exists:", path.exists())
    assert path.exists(), f"Missing checkpoint: {path}"

DUMLO original  : /content/drive/MyDrive/T2ICount-assets/checkpoints/dumlo_retrain/pilot_1000x10/dumlo_1000x10/best_model_7.pth
  exists: True
Count-low       : /content/drive/MyDrive/T2ICount-assets/checkpoints/dumlo_retrain/lambda_count_01_1000x10/dumlo_lc01_1000x10/best_model_7.pth
  exists: True
OT-low          : /content/drive/MyDrive/T2ICount-assets/checkpoints/dumlo_retrain/lambda_ot_001_1000x10/dumlo_lot001_1000x10/best_model_7.pth
  exists: True
Baseline        : /content/drive/MyDrive/T2ICount-assets/checkpoints/baseline_retrain/pilot_1000x10/baseline_1000x10/best_model_7.pth
  exists: True


NameError: name 'run_diagnostic' is not defined

## 7. Runner

In [12]:
def run_gradient_diagnostic(
    name,
    checkpoint,
    lambda_count,
    lambda_ot,
    lambda_tv=0.01,
    num_samples=8,
):
    log_path = OUT_DIR / f"{name}.log"

    cmd = [
        sys.executable,
        "-u",
        "tools/diagnose_dumlo_gradients.py",

        "--asset-root", ASSET_ROOT,
        "--checkpoint", str(checkpoint),

        "--train-samples", "1000",
        "--train-subset-seed", "3407",
        "--seed", "3407",
        "--num-samples", str(num_samples),

        "--dumlo-lambda-count", str(lambda_count),
        "--dumlo-lambda-ot", str(lambda_ot),
        "--dumlo-lambda-tv", str(lambda_tv),

        "--dumlo-epsilon", "10",
        "--dumlo-iters", "100",
        "--dumlo-aug-points", "10",
        "--dumlo-radius-factor", "0.5",
        "--dumlo-sampling-seed", "3407",
    ]

    print("=" * 100)
    print("EXPERIMENT:", name)
    print("CHECKPOINT:", checkpoint)
    print(
        f"lambda_count={lambda_count}, "
        f"lambda_ot={lambda_ot}, "
        f"lambda_tv={lambda_tv}"
    )
    print("NUM SAMPLES:", num_samples)
    print("LOG:", log_path)
    print()
    print("COMMAND:")
    print(" ".join(map(str, cmd)))
    print("=" * 100)
    print()

    start = time.perf_counter()

    with log_path.open("w", encoding="utf-8") as f:
        process = subprocess.Popen(
            cmd,
            cwd=REPO_ROOT,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )

        for line in process.stdout:
            print(line, end="", flush=True)
            f.write(line)
            f.flush()

        return_code = process.wait()

    elapsed = time.perf_counter() - start

    print()
    print("RETURN CODE:", return_code)
    print(f"TIME: {elapsed:.1f} sec")

    if return_code != 0:
        raise RuntimeError(
            f"{name} failed with return code {return_code}"
        )

    return {
        "name": name,
        "return_code": return_code,
        "elapsed_sec": elapsed,
        "log": str(log_path),
    }

In [15]:
# ============================================================
# Phase B:
# SAME diagnostic objective, DIFFERENT trained checkpoints.
#
# This isolates the effect of the learned model state.
# ============================================================

ORIGINAL_LAMBDA_COUNT = 1.0
ORIGINAL_LAMBDA_OT = 0.1
ORIGINAL_LAMBDA_TV = 0.01

phase_b = [
    # Original DUMLO was already measured earlier.
    # We skip rerunning it because:
    #
    # dumlo_e7_original_weights.log
    #
    # already uses exactly these weights and the same 8 samples.

    {
        "name": "countlow_e7_original_objective_probe",
        "checkpoint": CKPT_COUNT_LOW,
    },
    {
        "name": "otlow_e7_original_objective_probe",
        "checkpoint": CKPT_OT_LOW,
    },
    {
        "name": "baseline_e7_original_objective_probe",
        "checkpoint": CKPT_BASELINE,
    },
]

phase_b_results = []

for exp in phase_b:
    result = run_gradient_diagnostic(
        name=exp["name"],
        checkpoint=exp["checkpoint"],

        # IMPORTANT:
        # Same diagnostic objective for every checkpoint.
        lambda_count=ORIGINAL_LAMBDA_COUNT,
        lambda_ot=ORIGINAL_LAMBDA_OT,
        lambda_tv=ORIGINAL_LAMBDA_TV,

        num_samples=8,
    )

    phase_b_results.append(result)

print("\n" + "=" * 100)
print("PHASE B COMPLETE")
print("=" * 100)

for result in phase_b_results:
    print(
        f"{result['name']:40s} "
        f"rc={result['return_code']} "
        f"time={result['elapsed_sec']:.1f}s"
    )

print("\nLogs saved to:")
print(OUT_DIR)

EXPERIMENT: countlow_e7_original_objective_probe
CHECKPOINT: /content/drive/MyDrive/T2ICount-assets/checkpoints/dumlo_retrain/lambda_count_01_1000x10/dumlo_lc01_1000x10/best_model_7.pth
lambda_count=1.0, lambda_ot=0.1, lambda_tv=0.01
NUM SAMPLES: 8
LOG: /content/drive/MyDrive/T2ICount-assets/outputs/dumlo_gradient_diagnostics/countlow_e7_original_objective_probe.log

COMMAND:
/usr/bin/python3 -u tools/diagnose_dumlo_gradients.py --asset-root /content/T2ICount-assets --checkpoint /content/drive/MyDrive/T2ICount-assets/checkpoints/dumlo_retrain/lambda_count_01_1000x10/dumlo_lc01_1000x10/best_model_7.pth --train-samples 1000 --train-subset-seed 3407 --seed 3407 --num-samples 8 --dumlo-lambda-count 1.0 --dumlo-lambda-ot 0.1 --dumlo-lambda-tv 0.01 --dumlo-epsilon 10 --dumlo-iters 100 --dumlo-aug-points 10 --dumlo-radius-factor 0.5 --dumlo-sampling-seed 3407

Loading model from /content/T2ICount-assets/pretrained/sd-v1-5/v1-5-pruned-emaonly.ckpt
LatentDiffusion: Running in eps-prediction mod

## 8. Experiment

In [13]:
experiments = [
    {
        "name": "dumlo_e7_original_weights",
        "lambda_count": 1.0,
        "lambda_ot": 0.1,
    },
    {
        "name": "dumlo_e7_countlow_probe",
        "lambda_count": 0.1,
        "lambda_ot": 0.1,
    },
    {
        "name": "dumlo_e7_otlow_probe",
        "lambda_count": 1.0,
        "lambda_ot": 0.01,
    },
]

results = []

for exp in experiments:
    result = run_gradient_diagnostic(
        name=exp["name"],
        checkpoint=DUMLO_E7,
        lambda_count=exp["lambda_count"],
        lambda_ot=exp["lambda_ot"],
        lambda_tv=0.01,
        num_samples=8,
    )
    results.append(result)

print("\n" + "=" * 100)
print("DONE")
print("=" * 100)

for result in results:
    print(
        f"{result['name']:32s} "
        f"rc={result['return_code']} "
        f"time={result['elapsed_sec']:.1f}s"
    )

print("\nLogs:")
print(OUT_DIR)

EXPERIMENT: dumlo_e7_original_weights
CHECKPOINT: /content/drive/MyDrive/T2ICount-assets/checkpoints/dumlo_retrain/pilot_1000x10/dumlo_1000x10/best_model_7.pth
lambda_count=1.0, lambda_ot=0.1, lambda_tv=0.01
NUM SAMPLES: 8
LOG: /content/drive/MyDrive/T2ICount-assets/outputs/dumlo_gradient_diagnostics/dumlo_e7_original_weights.log

COMMAND:
/usr/bin/python3 -u tools/diagnose_dumlo_gradients.py --asset-root /content/T2ICount-assets --checkpoint /content/drive/MyDrive/T2ICount-assets/checkpoints/dumlo_retrain/pilot_1000x10/dumlo_1000x10/best_model_7.pth --train-samples 1000 --train-subset-seed 3407 --seed 3407 --num-samples 8 --dumlo-lambda-count 1.0 --dumlo-lambda-ot 0.1 --dumlo-lambda-tv 0.01 --dumlo-epsilon 10 --dumlo-iters 100 --dumlo-aug-points 10 --dumlo-radius-factor 0.5 --dumlo-sampling-seed 3407

Loading model from /content/T2ICount-assets/pretrained/sd-v1-5/v1-5-pruned-emaonly.ckpt
LatentDiffusion: Running in eps-prediction mode
DiffusionWrapper has 859.52 M params.
making atten